# Llama 3 Tokenizer

**Useful Links:**

- [Tokenization Visualization](https://tiktokenizer.vercel.app/?model=meta-llama%2FMeta-Llama-3-8B) 
- [HuggingFace Byte-Pair Encoding tokenization explanation](https://huggingface.co/learn/llm-course/chapter6/5)

From paper: "We use a vocabulary with $\bold{128K}$ tokens. Our token vocabulary combines $\bold{100K}$ tokens from the **tiktoken tokenizer** with $\bold{28K}$ additional tokens to better support non-English languages. Compared to the Llama $2$ tokenizer, our new tokenizer improves compression rates on a sample of English data from $3.17$ to $3.94$ characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding $\bold{28K}$ tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

- They took OpenAI's pre-trained tokenizer, then they got their own dataset consisting of non-English languages and code, ran a BPE training algorithm on that dataset to generate an additional 28K BPE merge rules, and finally combined the pre-trained tokenizer with the 28K tokens to get the Llama 3 tokenizer.

Unfortunately for my scaled down model I can not use OpenAI's pre-trained tokenizer, so I will need to train a new BPE tokenizer.

**Goals:**

- [x] Use HuggingFace's Tokenizer library to create a tokenizer that will be trained on a the FineWeb-edu dataset.

In [1]:
# @i-c
%load_ext autoreload
%autoreload 2

In [2]:
import EasyJupyter
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing
import os

In [3]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_config import BaseConfig

In [4]:
class BPETokenizer:
    def __init__(self, cfg: BaseConfig):
        """Handles configuring, training, saving and loading of the BPE tokenizer"""
        self.cfg = cfg

    def train_tokenizer(self, training_file_path: str) -> Tokenizer:
        """Train the tokenizer"""
        cfg = self.cfg
        special_tokens = cfg.special_tokens

        save_path = (
            cfg.MODEL_DIR / "saved_models" / "tokenizer" / "BPE_tokenizer_trained.json"
        )
        # Ensure the directory exists
        save_path.parent.mkdir(parents=True, exist_ok=True)

        # Initialize a new Byte-Pair-Encoding tokenizer model.
        tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]))

        # Using Byte-Level pre-tokenization ensures that spaces and punctuation are handled correctly without losing characters.
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

        # Configure the BPE trainer
        trainer = BpeTrainer(
            vocab_size=cfg.vocab_size,
            special_tokens=list(special_tokens.values()),
            initial_alphabet=ByteLevel.alphabet(),
        )

        # === Train the tokenizer on the raw text file
        print(f"Training the BPE tokenizer on {training_file_path}...")
        tokenizer.train(files=[training_file_path], trainer=trainer)

        # === Set up the post-processor to automatically add the begin_of_text token to sequences
        tokenizer.post_processor = TemplateProcessing(
            single=f"{special_tokens['doc_begin_token']} $A",
            pair=f"{special_tokens['doc_begin_token']} $A $B",
            special_tokens=[
                (
                    special_tokens["doc_begin_token"],
                    tokenizer.token_to_id(special_tokens["doc_begin_token"]),
                )
            ],
        )
        # Tell the tokenizer how to decode the token ids back into text, example: 'Ġ' back to ' '
        tokenizer.decoder = ByteLevelDecoder()

        # === Save the trained tokenizer to a JSON file
        tokenizer.save(str(save_path))
        print(f"Tokenizer saved to {save_path}")
        return tokenizer

    def load_tokenizer(self) -> tuple[bool, Tokenizer]:
        """Load the tokenizer"""
        tokenizer_filename = "BPE_tokenizer_trained.json"
        tokenizer_path = (
            self.cfg.MODEL_DIR / "saved_models" / "tokenizer" / tokenizer_filename
        )

        if os.path.exists(tokenizer_path):
            print(
                f"\nLoading existing trained BPE Tokenizer from: ({tokenizer_path})..."
            )
            tokenizer = Tokenizer.from_file(str(tokenizer_path))
            return True, tokenizer
        return False, None

In [5]:
# @i-c

from llama_config import Llama3_scaled_down
cfg = Llama3_scaled_down()
cfg._setup_folders()
t = BPETokenizer(cfg)

sample_training_text_path = "sample_dataset.txt"
with open(sample_training_text_path, "w", encoding="utf-8") as f:
    f.write("This is a test document for the Llama tokenizer!\n")
    f.write("Machine learning is awesome\n")

# Test: Train Tokenizer
print("\n\nTesting training a tokenizer...")
trained_tokenizer = t.train_tokenizer(sample_training_text_path)

print("\n\nTesting loading a trained tokenizer...")
success, loaded_tokenizer = t.load_tokenizer()

# Test: Loading A Trained Tokenizer
if success:
    print("Tokenizer successfully trained!")

    print("\n\nEncoding a string to IDs")
    text_str = "The brown rabbit ate the apple"
    encoded = loaded_tokenizer.encode(text_str)
    print(f"Original string: {text_str}")
    print(f"Token IDs: {encoded.ids}")
    print(f"Tokens: {encoded.tokens}")

    print("\n\nDecoding IDs to a string")
    decoded = loaded_tokenizer.decode(encoded.ids)
    print(f"Decoded string: {decoded}")



Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM
/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model


Testing training a tokenizer...
Training the BPE tokenizer on sample_dataset.txt...


Tokenizer saved to /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/tokenizer/BPE_tokenizer_trained.json


Testing loading a trained tokenizer...

Loading existing trained BPE Tokenizer from: (/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/tokenizer/BPE_tokenizer_trained.json)...
Tokenizer successfully trained!


Encoding a string to IDs
Original string: The brown rabbit ate the apple
Token IDs: [1, 269, 72, 224, 69, 85, 82, 90, 81, 224, 85, 68, 69, 69, 76, 87, 265, 87, 72, 293, 265, 83, 83, 79, 72]
Tokens: ['<|begin_of_doc|>', 'Th', 'e', 'Ġ', 'b', 'r', 'o', 'w', 'n', 'Ġ', 'r', 'a', 'b', 'b', 'i', 't', 'Ġa', 't', 'e', 'Ġthe', 'Ġa', 'p', 'p', 'l', 'e']


Decoding IDs to a string
Decoded string: Th